<h1 style="font-size: 1.6rem; font-weight: bold">ITO5201 Machine Learning</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Elements of Machine Learning: Stochastic Gradient Descent</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Stochastic Gradient Descent (SGD)**

**Goal:** Find the weight vector `w` that makes the model's predictions as close as possible to the true targets.

Imagine standing on a hilly landscape where your position is the current weight vector and the height is the error. SGD walks downhill — but instead of looking at the whole dataset to decide the direction (expensive), it looks at **one training example at a time** and takes a small step based on that alone.

**The recipe:**
1. **Initialize** pick a starting weight vector, e.g. $\mathbf{w}^{(0)} = [0, 0]$
2. **Get the gradient** compute the partial derivatives of the loss w.r.t. each weight, for ONE data point
3. **Decide a learning rate** $\eta$ controls the step size (here $\eta = 1$)
4. **Update** step in the opposite direction of the gradient, then repeat with the next point

<br>

### **Loss Function & Gradient**

#### **The model**
We predict with a linear model using the feature map $\varphi(x) = [1, x]$
(the 1 acts as a bias term):

$$\hat{y} = \mathbf{w} \cdot \varphi(x_n) = w_1 \cdot 1 + w_2 \cdot x_n$$

#### **The loss (per example)**
How wrong are we on a single point? Square the difference so positive and
negative errors both count:

$$E(\mathbf{w}) = \big[t_n - \mathbf{w} \cdot \varphi(x_n)\big]^2$$

Where:
- $t_n$ = the true target for example $n$
- $x_n$ = the input for example $n$
- $\varphi(x_n)$ = the feature vector $[1, x_n]$
- $\mathbf{w} \cdot \varphi(x_n)$ = the model's prediction

#### **The gradient**
Differentiate using the chain rule — bring down the power 2, multiply by
the derivative of the inside (which is $-\varphi(x_n)$):

$$\nabla E = -2\big[t_n - \mathbf{w} \cdot \varphi(x_n)\big]\varphi(x_n)$$

The gradient points **uphill** (toward more error), so we move against it.

#### **The update rule**

$$\mathbf{w}^{(new)} = \mathbf{w}^{(old)} - \eta \, \nabla E$$

<br>

*** 

### **Worked Example**

**Dataset:** $D = \{(\tfrac{1}{2},\, 3),\ (0,\, \tfrac{3}{2}),\ (-1,\, 1)\}$ — each pair is $(x_n, t_n)$

**Setup:** $\mathbf{w}^{(0)} = [0, 0]$, learning rate $\eta = 1$, features $\varphi(x) = [1, x]$

#### **Step 1: point $(\tfrac{1}{2}, 3)$**

Features: $\varphi = [1, \tfrac{1}{2}]$

Prediction: $\hat{y} = 0 \cdot 1 + 0 \cdot \tfrac{1}{2} = 0$

Gradient:
$$\nabla E = -2(3 - 0)[1, \tfrac{1}{2}] = [-6, -3]$$

Update:
$$\mathbf{w}^{(1)} = [0, 0] - [-6, -3] = [6, 3]$$

#### **Step 2: point $(0, \tfrac{3}{2})$**

Features: $\varphi = [1, 0]$

Prediction: $\hat{y} = 6 \cdot 1 + 3 \cdot 0 = 6$

Gradient:
$$\nabla E = -2(\tfrac{3}{2} - 6)[1, 0] = -2(-\tfrac{9}{2})[1, 0] = [9, 0]$$

Update:
$$\mathbf{w}^{(2)} = [6, 3] - [9, 0] = [6-9,\ 3-0] = [-3, 3]$$

#### **Step 3: point $(-1, 1)$**

Features: $\varphi = [1, -1]$

Prediction: $\hat{y} = -3 \cdot 1 + 3 \cdot (-1) = -6$

Gradient:
$$\nabla E = -2(1 - (-6))[1, -1] = -2(7)[1, -1] = [-14, 14]$$

Update:
$$\mathbf{w}^{(3)} = [-3, 3] - [-14, 14] = [11, -11]$$

#### **Why do the weights jump around so much?**
With $\eta = 1$ every step fully overcorrects for one example, so the
weights swing wildly. In practice we use a small learning rate
(e.g. $\eta = 0.01$) and many passes over the data, so each step nudges
the weights gently toward a good solution.

*** 

#### **What happens after Step 3?**
We've now used every point in the dataset once — that's called one
**epoch**. But the weights are almost never correct after a single epoch,
so we simply **repeat**: go back to the first data point with the current
weights $\mathbf{w}^{(3)} = [11, -11]$ and keep cycling through the data.

**The loop:**
1. Pick the next data point (wrap around to the start after the last one)
2. Compute the gradient using the current weights
3. Update the weights
4. Repeat until **convergence**

#### **When do we stop? (Convergence)**
We say SGD has **converged** when continuing no longer helps. Common
stopping conditions:

| Condition | Meaning |
|---|---|
| Weights barely change | $\|\mathbf{w}^{(new)} - \mathbf{w}^{(old)}\|$ is below a tiny threshold |
| Loss barely changes | The error has flattened out — more steps don't reduce it |
| Max epochs reached | A safety cap, e.g. stop after 100 passes through the data |

#### **Important caveat about this example**
With $\eta = 1$, our worked example will **never** converge — each step
fully overcorrects for one data point, so the weights swing wildly forever
(notice they went $[6,3] \to [-3,3] \to [11,-11]$). Convergence requires a
**small learning rate** (e.g. $\eta = 0.01$), or one that **decays** over
time — big steps early to move fast, small steps later to settle into the
minimum.

**Key intuition:** each individual SGD step is noisy (it only sees one
data point), but averaged over many steps and epochs, the weights drift
toward the values that minimize the error over the *whole* dataset.

***

### **1. Linear Regression Foundations (Part A)**

**Goal:** Nail down the core building blocks — how a linear model is built,
scored, and interpreted probabilistically.

Before optimising anything, you need the vocabulary. A linear regression
model turns inputs into predictions by weighting transformed features, and
(when viewed probabilistically) treats each prediction as the centre of a
bell curve rather than a single hard number.

#### **Matched terms and definitions**

| # | Term | Definition (paraphrased) |
|---|---|---|
| 1 | **Basis functions** | Transformations applied to raw inputs to create features that get linearly combined in the model. |
| 2 | **Squared loss as negative log likelihood** | Using squared prediction error as the loss corresponds to the negative log likelihood of a normally distributed target. |
| 3 | **Linear regression model** | A model assuming a straight-line relationship between inputs and output, computed as a weighted sum of the inputs. |
| 4 | **Normal distribution** | A symmetric bell-shaped distribution centred on the mean, where values near the mean occur more often. |
| 5 | **Weight vector** | The coefficients multiplying each input feature to produce a prediction. |
| 6 | **Probabilistic regression model** | A model that outputs a probability distribution (often normal) for the target, capturing uncertainty rather than a single value. |

<br>

### **2. Optimisation & Training (Part B)**

**Goal:** Understand the machinery that actually *finds* the best weights.

Once we have a loss function, we need a way to minimise it. Some methods
solve it in one shot (normal equations); others walk downhill step by step
(gradient descent and its faster, noisier cousin, SGD). Several knobs —
learning rate, batch size, stopping rule — control how that walk behaves.

#### **Matched terms and definitions**

| # | Term | Definition (paraphrased) |
|---|---|---|
| 7 | **Stopping criterion** | The rule that decides when to halt optimisation — e.g. loss barely changes, or a max number of iterations is hit. |
| 8 | **Normal equations** | A closed-form solution derived by setting the derivative of the least-squares error to zero, giving the best-fit line directly. |
| 9 | **Stochastic gradient descent** | Updates weights using a randomly chosen subset of data to estimate the gradient, speeding up learning versus the full dataset. |
| 10 | **Gradient descent** | Minimises a function by repeatedly stepping in the direction opposite the gradient (steepest descent). |
| 11 | **Batch size** | How many data points are used per SGD iteration — trading off computation speed against gradient accuracy. |
| 12 | **Learning rate** | A parameter setting the step size in gradient descent, controlling how fast (or unstably) the model learns. |

#### **Quick contrast: the two "solve" strategies**

| | Normal equations | Gradient descent / SGD |
|---|---|---|
| How it works | Solves in one shot (algebra) | Iterates step by step |
| Speed on huge data | Slow / infeasible | Scales well |
| Needs learning rate? | No | Yes |

<br>

### **3. Regularisation (Part C)**

**Goal:** Stop the model from overfitting by discouraging oversized weights.

Left unchecked, a model can fit training data *too* well — memorising noise
instead of learning patterns. Regularisation adds a penalty for large
coefficients, shrinking them toward zero so the model stays simpler and
generalises better to new data.

#### **Matched terms and definitions**

| # | Term | Definition (paraphrased) |
|---|---|---|
| 13 | **Lasso regression** | Uses an L1 penalty to reduce overfitting *and* perform feature selection by driving some coefficients exactly to zero. |
| 14 | **Regularisation parameter** | A scaling factor that adjusts how strongly the penalty term affects the loss — i.e. how much regularisation is applied. |
| 15 | **Ridge regression** | A linear regression variant with an L2 penalty on coefficient magnitude to curb complexity and overfitting. |
| 16 | **Penalty term** | The extra term added to the loss that punishes large parameter values, improving generalisation. |
| 17 | **Parameter shrinkage** | The process of reducing the magnitude of model parameters via regularisation to prevent overfitting. |

#### **Ridge vs Lasso at a glance**

| | Ridge | Lasso |
|---|---|---|
| Penalty type | L2 (squared weights) | L1 (absolute weights) |
| Effect on weights | Shrinks toward zero (rarely *to* zero) | Can set some exactly to zero |
| Feature selection? | No | Yes |

**Key intuition:** the **penalty term** is the "tax" on big weights, the
**regularisation parameter** sets the tax rate, and **parameter shrinkage**
is the result — smaller, tamer coefficients that generalise better.